# Basement membrane v1 — collagen IV network modulus & remodelling for the AICS vertex model

_Investigation `basement-membrane-v1` — coder reproduction notebook._

**Question.** What effective elastic modulus and remodelling (stress-relaxation) time does a coarse-grained collagen IV basement-membrane network produce on a growing (biaxially stretching) boundary — the two numbers the AICS 3D vertex model needs to couple to a basement membrane?

The first Vivarium build of the AICS lumenoid basement membrane: a coarse- grained collagen IV network on a moving boundary, run until it reports an elastic modulus and a remodelling time for the vertex model.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-lumenoid/viva-lumenoid').is_dir():
    REPO = Path('/home/runner/work/viva-lumenoid/viva-lumenoid')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_lumenoid.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: `bm-v1-stage1-modulus-remodelling`

**Purpose.** A crosslinked semiflexible rod network stores elastic energy in stretched bonds under fast load (crosslinks frozen), and dissipates it over the crosslink turnover time when make/break is switched on.

**Claim.** A coarse-grained collagen IV network (two-bead rods, stochastic NC1/7S crosslinks, no excluded volume) run through the spec's stage-1 protocol — stretch, hold, then switch the chemistry on — yields a separable elastic modulus and remodelling (stress-relaxation) time on real LAMMPS.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `collagen-iv-network-stage1` | `viva_lumenoid.composites.stage1_composite` | 0 | n_rods=200, box_xy=20.0, slab_thickness=2.0, bead_spacing_nm=114.0, persistence_length_nm=40.0, make_prob=0.12, make_every=100, break_every=100, factor_mult=0.66, target_strain=0.15, strain_rate=0.0015, assemble_steps=8000, hold_steps=16000, seed=12345 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_lumenoid.composites.stage1_composite`** — `spec_viva_lumenoid_composites_stage1_composite` (a plain, editable dict)


_composite spec file for `viva_lumenoid.composites.stage1_composite` not found under `viva_lumenoid/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: bm-v1-stage1-modulus-remodelling ===
STUDY = 'bm-v1-stage1-modulus-remodelling'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**cross-check**


In [ ]:
# cross-check
show_viz(_render_one('html:cross_check.html', {'title': 'Cross-check — clean-room vs the released code (headline)'}, RUNS_DB, STUDY_YAML))

**modulus-vs-fraction**


In [ ]:
# modulus-vs-fraction
show_viz(_render_one('html:modulus_vs_fraction.html', {'title': 'The faithful model reproduces the published modulus (vs crosslink fraction)'}, RUNS_DB, STUDY_YAML))

**modulus-ladder**


In [ ]:
# modulus-ladder
show_viz(_render_one('html:modulus_ladder.html', {'title': 'Modulus ladder — v1 in the published very-soft regime'}, RUNS_DB, STUDY_YAML))

**stage1-movie**


In [ ]:
# stage1-movie
show_viz(_render_one('html:stage1_movie.html', {'title': 'Simulation movie — the staged protocol unfolding'}, RUNS_DB, STUDY_YAML))

**stage1-movie-3d**


In [ ]:
# stage1-movie-3d
show_viz(_render_one('html:stage1_movie_3d.html', {'title': '3D view — the quasi-2D slab, rotatable'}, RUNS_DB, STUDY_YAML))

**stage1-faithful**


In [ ]:
# stage1-faithful
show_viz(_render_one('html:stage1_faithful.html', {'title': 'Stage 1 (faithful angles) — modulus AND remodelling separated'}, RUNS_DB, STUDY_YAML))

**stage1-diagnostic**


In [ ]:
# stage1-diagnostic
show_viz(_render_one('html:stage1_diagnostic.html', {'title': 'Stage 1 (angle-off) — modulus & remodelling diagnostic'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| network-assembles | kind=count field=n_crosslinks when=end-of-assemble | op gt value 0 field n_crosslinks |
| elastic-modulus-consistent-with-floppy | kind=slope field=sigma_inplane versus=strain phase=stretch ensemble=6 | op consistent_with_zero note floppy network; z < 4 |
| relaxation-decays-not-densifies | kind=delta field=sigma_inplane phase=hold_viscous compare_to=crosslink_count | op decreases note crosslinks decay 46→3; stress drops in the correct direction |
| remodelling-time-calibratable | kind=relaxation_time field=sigma_inplane phase=hold_viscous convert=hours | op calibratable target_hours 20 range_hours 3-30 across systems |


## Study: `bm-v2-stress-vs-strainrate`

**Purpose.** At ε̇ below the crosslink turnover rate the network remodels fast enough to flow (σ ∝ ε̇, constant η); above it, bonds cannot relax between deformation increments and the response stiffens.

**Claim.** Imposing an equibiaxial in-plane strain rate ε̇ on the growing collagen IV network and reading the steady-state stress σ(ε̇) gives the effective viscosity η = σ/ε̇ (viscous at slow rates) and the rate at which the response turns elastic — the σ(ε̇) relation the vertex model needs.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `collagen-iv-network-stage2` | `viva_lumenoid.composites.stage2_sweep` | 0 | n_rods=150, box_xy=18.0, assemble_steps=4000, make_prob=0.12, strain_rates=[5e-4, 1e-3, 2e-3, 4e-3, 8e-3, 1.6e-2], seed=12345 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_lumenoid.composites.stage2_sweep`** — `spec_viva_lumenoid_composites_stage2_sweep` (a plain, editable dict)


_composite spec file for `viva_lumenoid.composites.stage2_sweep` not found under `viva_lumenoid/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: bm-v2-stress-vs-strainrate ===
STUDY = 'bm-v2-stress-vs-strainrate'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**coupling**


In [ ]:
# coupling
show_viz(_render_one('html:coupling.html', {'title': 'Vertex-model coupling — E(ε̇) and τ(ε̇) (the deliverable)'}, RUNS_DB, STUDY_YAML))

**stage2-movie**


In [ ]:
# stage2-movie
show_viz(_render_one('html:stage2_movie.html', {'title': 'Simulation movie — the growing substrate stretches'}, RUNS_DB, STUDY_YAML))

**stage2-movie-3d**


In [ ]:
# stage2-movie-3d
show_viz(_render_one('html:stage2_movie_3d.html', {'title': '3D view — the slab stretching, rotatable'}, RUNS_DB, STUDY_YAML))

**stage2-diagnostic**


In [ ]:
# stage2-diagnostic
show_viz(_render_one('html:stage2_diagnostic.html', {'title': 'Stage 2 — σ(ε̇) diagnostic'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| sweep-runs | kind=count field=rates | op gt value 0 field n_rates |
| viscosity-defined | kind=value field=eta_lowrate | op finite field eta_lowrate |
| viscous-plateau-vs-elastic | kind=slope field=eta_effective versus=strain_rate scale=log | op plateau note η ~ const at slow ε̇ |


## Study: `bm-v3-junction-bending-rigidity`

**Purpose.** Maxwell rigidity percolation: a central-force network is floppy below z = 2d (= 4 in 2D). Bond-bending constraints lower the threshold and rigidify sub- isostatic networks.

**Claim.** SUPERSEDED by the 2026-10-01 cross-check. This study originally explained the soft modulus as a central-force rigidity-percolation effect (z ≤ 3 < 4). The released code is NOT central-force — it sets junction angles (NC1 180° k=4.0, 7S 155°/60°) — so that explanation does not describe the published model, which carries those angles and is still soft (≈0.03 Pa). What this sweep actually shows: with angles OFF (our earlier transcription) and the corrected params, the network is sub-isostatic (z tops out ≈2.0) and soft — a property of the angle- off transcription, not of the paper.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `connectivity-sweep` | `viva_lumenoid.composites.connectivity_sweep` | 0 | n_rods=200, box_xy=20.0, assemble_steps=8000, hold_steps=8000, rungs=[(1,3,0.12),(2,4,0.25),(3,5,0.40),(5,8,0.60)], seeds=[11,22,33] |

Declared parameter sets (`study.yaml` variants):

- **nc1x1-7sx3** — `nc1_max_crosslinks=1`, `svns_max_crosslinks=3`, `make_prob=0.12`
- **nc1x2-7sx4** — `nc1_max_crosslinks=2`, `svns_max_crosslinks=4`, `make_prob=0.25`
- **nc1x3-7sx5** — `nc1_max_crosslinks=3`, `svns_max_crosslinks=5`, `make_prob=0.4`
- **nc1x5-7sx8** — `nc1_max_crosslinks=5`, `svns_max_crosslinks=8`, `make_prob=0.6`


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_lumenoid.composites.connectivity_sweep`** — `spec_viva_lumenoid_composites_connectivity_sweep` (a plain, editable dict)


_composite spec file for `viva_lumenoid.composites.connectivity_sweep` not found under `viva_lumenoid/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: bm-v3-junction-bending-rigidity ===
STUDY = 'bm-v3-junction-bending-rigidity'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**rigidity-movie**


In [ ]:
# rigidity-movie
show_viz(_render_one('html:rigidity_movie.html', {'title': 'Simulation movie — high-connectivity network (NC1×5 + 7S×8)'}, RUNS_DB, STUDY_YAML))

**rigidity-movie-3d**


In [ ]:
# rigidity-movie-3d
show_viz(_render_one('html:rigidity_movie_3d.html', {'title': '3D view — the connected network, rotatable'}, RUNS_DB, STUDY_YAML))

**rigidity-diagnostic**


In [ ]:
# rigidity-diagnostic
show_viz(_render_one('html:rigidity_diagnostic.html', {'title': 'Rigidity sweep — modulus vs coordination z'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| z-stays-below-threshold | kind=max field=coordination_z | op lt value 4.0 field max_z |
| modulus-stays-floppy | kind=value field=elastic_modulus_vs_z ensemble=3 | op consistent_with_zero note all CIs span 0 |


## Study: `bm-v4-junction-bending`

**Purpose.** Bending constraints at crosslink junctions lift a sub-isostatic (z < 4) central-force network out of the floppy regime — the physically-expected rigidity source for a semiflexible fiber network.

**Claim.** Junction bending is a released feature (NC1 angle k=4.0 @180°, plus 7S angles), not a new ingredient. HOW it is applied decides whether it yields a usable modulus. Our original as-formed `fix restrain` (pinned on AFTER assembly) is ill-conditioned — sign-flipping, no reproducible value. The FAITHFUL path (2026-10-02), creating real harmonic angles as the crosslinks form so the network equilibrates with them active (matching the released bond/react templates), yields a POSITIVE, well-conditioned, reproducible modulus. So the released model is not floppy once the angles are faithfully active; its modulus is finite and strongly density-dependent.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `junction-bending-network` | `viva_lumenoid.composites.bending_sweep` | 0 | n_rods=200, box_xy=20.0, bending_k=4.0, timestep=0.002, seed=12345 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_lumenoid.composites.bending_sweep`** — `spec_viva_lumenoid_composites_bending_sweep` (a plain, editable dict)


_composite spec file for `viva_lumenoid.composites.bending_sweep` not found under `viva_lumenoid/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: bm-v4-junction-bending ===
STUDY = 'bm-v4-junction-bending'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**faithful-angles**


In [ ]:
# faithful-angles
show_viz(_render_one('html:faithful_angles.html', {'title': 'Faithful angles — the modulus becomes reproducible (headline)'}, RUNS_DB, STUDY_YAML))

**bending-movie**


In [ ]:
# bending-movie
show_viz(_render_one('html:bending_movie.html', {'title': 'Simulation movie — rods pinned at crosslink angles'}, RUNS_DB, STUDY_YAML))

**bending-movie-3d**


In [ ]:
# bending-movie-3d
show_viz(_render_one('html:bending_movie_3d.html', {'title': '3D view — the bending-pinned network, rotatable'}, RUNS_DB, STUDY_YAML))

**bending-diagnostic**


In [ ]:
# bending-diagnostic
show_viz(_render_one('html:bending_diagnostic.html', {'title': 'FP1 — junction bending response'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| floppy-baseline-clean | kind=value field=modulus_vs_bending_k at=bending_k=0 | op near_zero tol 0.1 |
| bending-changes-mechanics | kind=magnitude field=modulus_vs_bending_k at=bending_k>0 | op gt value 1.0 note |E| jumps orders of magnitude above floppy |
| reproducible-modulus | kind=iqr_over_median field=modulus_vs_bending_k at=bending_k>0 | op lt value 0.5 note IQR < 0.5·|median| |


## Study: `bm-v5-porosity-bundling`

**Purpose.** Porosity is set by the areal density of rods (an emergent mesh); bundling requires lateral association of 5–7 protomers into a strand, a bond the released NC1/7S model does not have (it can bundle only through 7S branching).

**Claim.** Of the two geometric readouts the spec asks the network to reproduce, one is met and one is not: the simulated pore-size distribution reaches the measured corneal-EM band (72 ± 40 nm) at realistic density, while the protomers-per- strand distribution stays near 1, below the measured 5–7. The released NC1/7S topology has no lateral-association bond, so it cannot form multi-protomer strands — a limitation the spec identifies in advance.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `porosity-bundling-network` | `viva_lumenoid.composites.porosity_bundling` | 0 | n_rods=500, box_xy=20.0, assemble_steps=8000 |

Declared parameter sets (`study.yaml` variants):

- **density-sweep** — `box_xy=20.0`


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_lumenoid.composites.porosity_bundling`** — `spec_viva_lumenoid_composites_porosity_bundling` (a plain, editable dict)


_composite spec file for `viva_lumenoid.composites.porosity_bundling` not found under `viva_lumenoid/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: bm-v5-porosity-bundling ===
STUDY = 'bm-v5-porosity-bundling'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**assembly-movie**


In [ ]:
# assembly-movie
show_viz(_render_one('html:assembly_movie.html', {'title': 'Simulation movie — a porous collagen IV mesh forms'}, RUNS_DB, STUDY_YAML))

**porosity-bundling**


In [ ]:
# porosity-bundling
show_viz(_render_one('html:porosity_bundling.html', {'title': 'Stage 5 — porosity & bundling vs measured BM'}, RUNS_DB, STUDY_YAML))

**evidence-map**


In [ ]:
# evidence-map
show_viz(_render_one('html:evidence_map.html', {'title': 'Evidence map — readouts vs measured comparators'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| porosity-reaches-EM-band | kind=median field=pore_diameter | op between lo 32.0 hi 112.0 field median_pore_nm |
| bundling-under-shoots-5-7 | kind=mean field=protomers_per_strand | op lt value 2.0 field mean_strand |


## Open decisions
- DONE 2026-10-01 — cross-checked against the released code AND corrected the clean-room parameters to match (bond k/r0, rod r0=3.0, cutoffs, damp=0.1, Nevery=500, σ≈125 nm); re-ran all studies. Result: very-soft modulus consistent with the published ≈0.03 Pa; central-force/bending explanations retired. REMAINING for a quantitative modulus: angle set active during assembly (bond/react), 7S break cutoff (0.95), dense initial config, larger network.
- Engine for stages 1–2: LAMMPS now; revisit Smoldyn at stage 3 (spec #2).
- Secretion: fixed concentration vs fixed flux (spec #3).
- Energy scale: report modulus in Pa (fit to a measured modulus) or keep kT and report a ratio (spec #4).
- Coupling contract: which vertex-model term receives the BM — basal tension, mobility, or elastic spring (spec #5).
- Linker values: density, spring constant, off-rate; sweep off-rate as a multiple of the crosslink break rate (spec #6).
- Growth rate ε̇ from the lumenoid radius time course — still requested (spec #7).
- Slab thickness / layer count — one z-stack away (spec #9).
- Which readout calibrates vs validates — remodelling time calibrates MP and the time unit, so it cannot also validate (spec #10).
